In [ ]:
import json
from pathlib import Path

import lmstudio as lms

## Instellingen en model laden

In [ ]:
MODEL_NAME = "google/gemma-4-e4b"
PLANNING_FILE = Path("planning.txt")

model = lms.llm(MODEL_NAME)

## input analyseren

Deze functie gebruikt het LLM om vrije tekst om te zetten naar gestructureerde JSON-data.

In [ ]:
def analyseer_student_input(gebruikersinput: str) -> dict:
    parser_chat = lms.Chat(
        """
Je bent een parser voor een studieplanner-agent.

Zet de input van de student om naar geldig JSON.
Geef ALLEEN JSON terug. Geen uitleg, geen markdown.

Gebruik exact dit formaat:
{
  "vakken": [
    {
      "naam": "AI",
      "deadline": "vrijdag",
      "moeilijkheid": 4
    }
  ],
  "tijdslots": [
    {
      "dag": "maandag",
      "uren": 2
    }
  ]
}

Regels:
- moeilijkheid is een getal van 1 t/m 5.
- Als moeilijkheid onbekend is, schat deze op basis van de tekst.
- Als deadline onbekend is, gebruik null.
- Als tijdslots onbekend zijn, gebruik een lege lijst.
"""
    )
  
    parser_chat.add_user_message(f"Studentinput:\n{gebruikersinput}")

    response = model.respond(parser_chat)
    raw_output = response.content.strip()

    try:
        return json.loads(raw_output)

    except json.JSONDecodeError:
        print("\n❌ De input kon niet goed worden omgezet naar JSON.")
        print("LLM output was:")
        print(raw_output)

        return {
            "vakken": [],
            "tijdslots": [],
        }

## Tijdslots bepalen

Deze functie haalt tijdslots uit de gestructureerde input.  
Als er geen tijdslots zijn gevonden, geeft de functie een lege lijst terug.


In [ ]:
def bereken_tijdslots(gestructureerde_data: dict) -> list:

    tijdslots = gestructureerde_data.get("tijdslots", [])

    if tijdslots:
        return tijdslots

    print("\n⚠️ Geen beschikbare studietijd gevonden in de input.")
    return []

## Input valideren

Hier controleren we of er genoeg informatie is om een planning te maken.

In [ ]:
def valideer_input(gestructureerde_data: dict, tijdslots: list) -> bool:
    vakken = gestructureerde_data.get("vakken", [])
    return bool(vakken) and bool(tijdslots)

## Tools voor de agent

Deze functies kunnen door het LLM worden aangeroepen via `model.act()`.  
De agent kan hiermee prioriteiten tonen, een planning opslaan en herinneringen simuleren.

In [ ]:
def sla_prioriteiten_op(prioriteiten_json: str) -> str:

    print(f"\n📋 Prioriteiten opgeslagen:\n{prioriteiten_json}")
    return "Prioriteiten opgeslagen."


def sla_planning_op(planning_tekst: str) -> str:

    PLANNING_FILE.write_text(planning_tekst, encoding="utf-8")

    print(f"\n📅 Planning weggeschreven naar {PLANNING_FILE}")
    return "Planning opgeslagen."

# Simuleert het sturen van een herinnering.
def stuur_herinnering(bericht: str, tijdstip: str) -> str:

    print(f"\n🔔 Herinnering ingepland voor {tijdstip}: {bericht}")
    return f"Herinnering ingepland voor {tijdstip}."

## Agent-instructies

Hier maken we de systeeminstructie voor de studieplanner-agent.

In [ ]:
def maak_agent_chat() -> lms.Chat:
    return lms.Chat(
        """
Je bent een slimme studieplanner-agent.

Je helpt studenten met het maken van een realistische studieplanning op basis van:
- vakken
- deadlines
- moeilijkheid
- beschikbare tijdslots

Werk stap voor stap:
1. Analyseer de vakken en deadlines.
2. Bepaal de prioriteit per vak.
3. Verdeel de beschikbare studie-uren logisch.
4. Maak een concrete weekplanning.
5. Plan herinneringen in via de beschikbare tool.

Gebruik altijd de tools om resultaten op te slaan.
"""
    )

## Prompt bouwen

Deze functie combineert de originele input, de gestructureerde data en de tijdslots tot één prompt.

In [ ]:
def bouw_planning_prompt(
    gebruikersinput: str,
    gestructureerde_data: dict,
    tijdslots: list
) -> str:
    return f"""
De student heeft dit ingevoerd:
{gebruikersinput}

De input is omgezet naar deze gestructureerde data:
{json.dumps(gestructureerde_data, ensure_ascii=False, indent=2)}

Beschikbare tijdslots:
{json.dumps(tijdslots, ensure_ascii=False, indent=2)}

Maak nu:
1. Een geprioriteerde vakkenlijst. Gebruik hiervoor de tool sla_prioriteiten_op.
2. Een concrete weekplanning. Gebruik hiervoor de tool sla_planning_op.
3. Twee nuttige herinneringen. Gebruik hiervoor de tool stuur_herinnering.

Let op:
- Gebruik de deadlines om prioriteiten te bepalen.
- Geef moeilijkere vakken meer studietijd.
- Houd rekening met de beschikbare tijdslots.
- Maak de planning praktisch en duidelijk.
"""

## het programma

Deze functie voert de volledige agent-workflow uit.

In [ ]:
def run_studie_agent() -> None:
    print(f"=== 📚 Slimme Studie-Agent ({MODEL_NAME}) ===\n")

    gebruikersinput = input(
        "Voer je vakken, deadlines, moeilijkheid en beschikbare tijd in vrije tekst in:\n> "
    )

    print("=== 📚 Slimme Studie-Agent (Is aan het nadenken!) ===\n")

    gestructureerde_data = analyseer_student_input(gebruikersinput)
    tijdslots = bereken_tijdslots(gestructureerde_data)

    print("\n📦 Gestructureerde data:")
    print(json.dumps(gestructureerde_data, ensure_ascii=False, indent=2))

    print("\n⏰ Beschikbare tijdslots:")
    print(json.dumps(tijdslots, ensure_ascii=False, indent=2))

    if not valideer_input(gestructureerde_data, tijdslots):
        print("\n❌ Niet genoeg informatie om een planning te maken.")
        print("Voeg minimaal één vak/deadline én beschikbare studietijd toe en probeer opnieuw.")
        return

    chat = maak_agent_chat()
    prompt = bouw_planning_prompt(
        gebruikersinput,
        gestructureerde_data,
        tijdslots
    )

    chat.add_user_message(prompt)

    print("\n🤖 Agent is bezig...\n")

    model.act(
        chat,
        tools=[
            sla_prioriteiten_op,
            sla_planning_op,
            stuur_herinnering
        ],
        on_message=chat.append,
        on_prediction_fragment=lambda fragment, *_: print(
            fragment.content,
            end="",
            flush=True
        ),
    )

    print("\n\n─────────────────────────────")
    feedback = input(
        "\n📝 Geef feedback op de planning, of druk Enter om te stoppen:\n> "
    )

    if feedback.strip():
        feedback_prompt = (
            f"De student geeft deze feedback op de planning:\n{feedback}\n\n"
            "Pas de planning aan op basis van deze feedback en sla de nieuwe planning opnieuw op."
        )

        chat.add_user_message(feedback_prompt)

        print("\n🔄 Planning aanpassen...\n")

        model.act(
            chat,
            tools=[
                sla_prioriteiten_op,
                sla_planning_op,
                stuur_herinnering
            ],
            on_message=chat.append,
            on_prediction_fragment=lambda fragment, *_: print(
                fragment.content,
                end="",
                flush=True
            ),
        )

    print(f"\n\n✅ Klaar! Bekijk {PLANNING_FILE} voor de definitieve planning.")

## Agent testen

Voer deze cel uit om de studieplanner-agent te starten.

Voorbeeldinput:

`Ik moet AI vrijdag af hebben en data analyse volgende week maandag. AI vind ik lastig, data analyse gemiddeld. Ik heb maandag 2 uur, dinsdag 3 uur en donderdag 4 uur tijd.`

In [ ]:
run_studie_agent()

## Planning bekijken

De planning wordt opgeslagen in `planning.txt`.  
Met deze cel kun je het resultaat direct in de notebook bekijken.

In [ ]:
print(PLANNING_FILE.read_text(encoding="utf-8"))